In [14]:
# %% [markdown]
# # HARLF — Historical Sentiment via GDELT BigQuery (2004 → Today)
# - Source: BigQuery public dataset `gdelt-bq.gdeltv2.gkg`
# - Sentiment fields: V2Tone -> [tone, positive, negative, ...]
# - Entity filters: matches in V2Orgs / V2Persons using company name or ticker (case-insensitive)
# - Output: raw rows + (optional) monthly rollups
#
# Prereqs:
# 1) A Google account. In **Colab**, just run the auth cell below.
# 2) Locally: set GOOGLE_APPLICATION_CREDENTIALS to your service-account JSON.
#
# Cost note: BigQuery free tier gives ~1 TB query processing/month. This is plenty for asset-level pulls.

# %%
!pip -q install google-cloud-bigquery pandas pyarrow tqdm python-dateutil

# %%
import os, re, json, math, datetime as dt
from typing import Dict, List, Tuple

import pandas as pd
from tqdm import tqdm
from dateutil.relativedelta import relativedelta

# BigQuery client
from google.cloud import bigquery

# If running in Colab, uncomment:
# try:
#     from google.colab import auth
#     auth.authenticate_user()
#     print("✅ Google account authenticated.")
# except Exception:
#     pass

# -----------------------------
# Config
# -----------------------------
OUT_DIR = "data/raw/news"
RAW_OUT_CSV = os.path.join(OUT_DIR, "raw_gkg_sentiment_2004_today.csv")
MONTHLY_OUT_CSV = os.path.join(OUT_DIR, "raw_gkg_sentiment_monthly.csv")
STATS_JSON = os.path.join(OUT_DIR, "raw_gkg_sentiment_stats.json")

QUALITY_DOMAINS = {
    "reuters.com","bloomberg.com","wsj.com","ft.com","cnbc.com","marketwatch.com",
    "barrons.com","economist.com","forbes.com","morningstar.com","seekingalpha.com",
    "yahoo.com","nytimes.com","washingtonpost.com","theguardian.com","investors.com"
}

PORTFOLIO_CSV_PATH = "portfolio_holdings.csv"  # use if available

# Earliest date to query (GKG 2.0 supports 2015+ reliably; earlier via GDELT 1.0).
# The BigQuery mirrored table `gdelt-bq.gdeltv2.gkg` contains long history; we'll still clip to 2004-01-01.
START_DATE = dt.date(2004, 1, 1)
END_DATE = dt.date.today()

# -----------------------------
# Default tickers if portfolio CSV absent
# -----------------------------
DEFAULT_TICKERS: Dict[str, str] = {
    "NVDA": "NVIDIA Corporation",
    "MSFT": "Microsoft Corporation",
    "GOOGL": "Alphabet Inc",
    "AMD": "Advanced Micro Devices",
    "RDDT": "Reddit Inc",
    "ASML": "ASML Holding NV",
    "AMZN": "Amazon.com Inc",
    "META": "Meta Platforms Inc",
    "AAPL": "Apple Inc",
    "TSLA": "Tesla Inc",
    "SMR": "NuScale Power Corporation",
    "MU": "Micron Technology Inc",
    "MRVL": "Marvell Technology Group",
    "PLUG": "Plug Power Inc",
    "IONQ": "IonQ Inc",
    "RGTI": "Rigetti Computing Inc",
    "ARBE": "Arbe Robotics Ltd",
}

# -----------------------------
# Helpers
# -----------------------------
def ensure_dir(path: str):
    os.makedirs(os.path.dirname(path), exist_ok=True)

def load_portfolio(path: str = PORTFOLIO_CSV_PATH, fallback: Dict[str, str] = None) -> Dict[str, str]:
    if fallback is None:
        fallback = DEFAULT_TICKERS
    try:
        if os.path.exists(path):
            df = pd.read_csv(path)
            cols = {c.lower(): c for c in df.columns}
            if "ticker" in cols and "company" in cols:
                tcol, ccol = cols["ticker"], cols["company"]
            else:
                t_candidates = [c for c in df.columns if c.lower().startswith("tick")]
                c_candidates = [c for c in df.columns if c.lower().startswith("comp")]
                tcol = t_candidates[0] if t_candidates else df.columns[0]
                ccol = c_candidates[0] if c_candidates else df.columns[1]
            mapping = {
                str(r[tcol]).strip().upper(): str(r[ccol]).strip()
                for _, r in df.iterrows() if pd.notna(r[tcol]) and pd.notna(r[ccol])
            }
            if mapping:
                print(f"✅ Loaded {len(mapping)} tickers from {path}")
                return mapping
    except Exception as e:
        print(f"⚠️ Could not load portfolio CSV: {e}")
    print("ℹ️ Using default tickers.")
    return fallback

def build_keyword_regex(company: str, ticker: str) -> str:
    """
    Build a conservative word-boundary regex for matching company OR ticker
    inside GKG entity fields (V2Orgs / V2Persons). Lowercased.
    """
    # Keep a compact alias list (extend here if you like)
    aliases = {company, ticker}
    # Add a simplified company name without suffixes to catch variations
    simple = re.sub(r'(?i)\b(inc|corp(oration)?|co(mpany)?|ltd|nv|plc|limited|group|holdings?)\b\.?', '', company).strip()
    if len(simple) >= 3:
        aliases.add(simple)
    # Clean and dedupe
    tokens = sorted({re.sub(r'\s+', ' ', a.strip()) for a in aliases if a})
    # Word-boundary regex OR-ed
    parts = []
    for a in tokens:
        # escape special chars but keep spaces -> '\s+'
        esc = re.escape(a.lower()).replace(r'\ ', r'\s+')
        parts.append(rf'\b{esc}\b')
    return "(" + "|".join(parts) + ")"

def parse_v2tone_to_cols():
    """
    BigQuery expression that splits V2Tone string:
    order: Tone, Positive, Negative, Polarity, ActivityRefDensity, SelfGroupDensity, WordCount
    """
    return """
      CAST(SPLIT(V2Tone, ',')[OFFSET(0)] AS FLOAT64) AS tone,
      CAST(SPLIT(V2Tone, ',')[OFFSET(1)] AS FLOAT64) AS positive,
      CAST(SPLIT(V2Tone, ',')[OFFSET(2)] AS FLOAT64) AS negative,
      CAST(SPLIT(V2Tone, ',')[OFFSET(3)] AS FLOAT64) AS polarity,
      CAST(SPLIT(V2Tone, ',')[OFFSET(6)] AS INT64)   AS word_count
    """

def bq_client() -> bigquery.Client:
    # Requires authentication. In Colab, the auth cell above is enough.
    return bigquery.Client()

def date_to_gkg_int(d: dt.date, end_of_day: bool = False) -> int:
    """
    GKG DATE is INT of form YYYYMMDDHHMMSS.
    We'll use 000000 for start-of-day and 235959 for end-of-day.
    """
    return int(d.strftime("%Y%m%d") + ("235959" if end_of_day else "000000"))

# -----------------------------
# Core BigQuery fetch
# -----------------------------
def fetch_gkg_for_asset(ticker: str, company: str,
                        start_date: dt.date, end_date: dt.date,
                        limit_per_month: int = None) -> pd.DataFrame:
    """
    Query BigQuery GKG for a single asset using org/person matches.
    Returns: DataFrame with columns:
      [ticker, date, title?, url, domain, source_common_name, tone, positive, negative, polarity, word_count, is_quality_source]
    Note: GKG does not reliably have article title; we return URL + domain.
    """
    client = bq_client()
    pattern = build_keyword_regex(company, ticker)  # lowercased regex

    start_key = date_to_gkg_int(start_date, end_of_day=False)
    end_key   = date_to_gkg_int(end_date,   end_of_day=True)

    # We match in V2Orgs or V2Persons. Use COALESCE to avoid NULL
    # We also ensure DATE range, and require V2Tone present.
    # Extract domain from URL; use SourceCommonName if available.
    # Optional LIMIT per month is implemented outside (we’ll fetch all; you can add per-month caps if needed).
    query = f"""
    SELECT
      @ticker AS ticker,
      DATETIME(TIMESTAMP(PARSE_DATETIME('%Y%m%d%H%M%S', CAST(DATE AS STRING))), 'UTC') AS date_utc,
      DocumentIdentifier AS url,
      REGEXP_EXTRACT(DocumentIdentifier, r'https?://([^/]+)') AS domain,
      SourceCommonName AS source_common_name,
      {parse_v2tone_to_cols()}
    FROM `gdelt-bq.gdeltv2.gkg`
    WHERE DATE BETWEEN @start_key AND @end_key
      AND V2Tone IS NOT NULL
      AND (
        REGEXP_CONTAINS(LOWER(COALESCE(V2Orgs,'')),    @pattern)
        OR REGEXP_CONTAINS(LOWER(COALESCE(V2Persons,'')), @pattern)
      )
    """

    job_config = bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter("ticker", "STRING", ticker),
            bigquery.ScalarQueryParameter("start_key", "INT64", start_key),
            bigquery.ScalarQueryParameter("end_key", "INT64", end_key),
            bigquery.ScalarQueryParameter("pattern", "STRING", pattern),
        ]
    )

    df = client.query(query, job_config=job_config).result().to_dataframe(create_bqstorage_client=False)
    if df.empty:
        return df

    # Clean / finalize
    df["date"] = pd.to_datetime(df["date_utc"], utc=True).dt.tz_convert(None)
    df.drop(columns=["date_utc"], inplace=True)
    # Quality flag
    df["domain"] = (df["domain"] or "").fillna("")
    df["domain"] = df["domain"].str.lower().str.replace("^www\\.", "", regex=True)
    df["is_quality_source"] = df["domain"].isin(QUALITY_DOMAINS)
    # GKG has no article title field; keep URL-based dedup
    df = df.drop_duplicates(subset=["url"]).sort_values("date").reset_index(drop=True)
    return df

# -----------------------------
# Run for entire portfolio
# -----------------------------
def collect_portfolio_sentiment(ticker_to_name: Dict[str,str],
                                start_date: dt.date = START_DATE,
                                end_date: dt.date = END_DATE) -> pd.DataFrame:
    all_rows = []
    for tkr, name in tqdm(ticker_to_name.items(), desc="Assets"):
        try:
            part = fetch_gkg_for_asset(tkr, name, start_date, end_date)
            if not part.empty:
                all_rows.append(part)
        except Exception as e:
            print(f"⚠️ {tkr}: query failed: {e}")
    if not all_rows:
        return pd.DataFrame(columns=[
            "ticker","date","url","domain","source_common_name",
            "tone","positive","negative","polarity","word_count","is_quality_source"
        ])
    df = pd.concat(all_rows, ignore_index=True)
    # Final tidy
    # Some rows may miss domain; try to re-extract robustly
    df["domain"] = df["domain"].fillna("").str.lower().str.replace("^www\\.", "", regex=True)
    df["is_quality_source"] = df["domain"].isin(QUALITY_DOMAINS)
    return df

def save_outputs(df_raw: pd.DataFrame):
    ensure_dir(RAW_OUT_CSV)
    df_raw.to_csv(RAW_OUT_CSV, index=False)
    print(f"💾 Saved raw rows: {RAW_OUT_CSV} ({len(df_raw):,} rows)")

    # Monthly rollup (mean tone, pos, neg) per ticker
    if df_raw.empty:
        pd.DataFrame(columns=["ticker","year","month","n","tone_mean","pos_mean","neg_mean","quality_ratio"]).to_csv(MONTHLY_OUT_CSV, index=False)
        with open(STATS_JSON, "w") as f:
            json.dump({"rows": 0}, f, indent=2)
        print("ℹ️ No rows; wrote empty rollups.")
        return

    df_raw["year"] = df_raw["date"].dt.year
    df_raw["month"] = df_raw["date"].dt.month
    grp = df_raw.groupby(["ticker","year","month"])
    roll = grp.agg(
        n=("url","count"),
        tone_mean=("tone","mean"),
        pos_mean=("positive","mean"),
        neg_mean=("negative","mean"),
        quality_ratio=("is_quality_source","mean"),
    ).reset_index()
    roll.to_csv(MONTHLY_OUT_CSV, index=False)
    print(f"💾 Saved monthly rollups: {MONTHLY_OUT_CSV} ({len(roll):,} rows)")

    stats = {
        "rows": int(len(df_raw)),
        "date_min": df_raw["date"].min().isoformat() if len(df_raw) else None,
        "date_max": df_raw["date"].max().isoformat() if len(df_raw) else None,
        "by_ticker": df_raw.groupby("ticker")["url"].count().to_dict(),
        "quality_ratio_overall": round(float(df_raw["is_quality_source"].mean()) if len(df_raw) else 0.0, 4),
    }
    with open(STATS_JSON, "w") as f:
        json.dump(stats, f, indent=2)
    print(f"📊 Stats: {STATS_JSON}")

# -----------------------------
# Execute
# -----------------------------
tickers = load_portfolio()
print(f"Assets: {list(tickers.keys())}")

df_raw = collect_portfolio_sentiment(tickers, START_DATE, END_DATE)
save_outputs(df_raw)

print("\n✅ Done. Next step: feed monthly rollups into your HARLF sentiment pipeline.")


✅ Loaded 18 tickers from portfolio_holdings.csv
Assets: ['RDDT', 'NVDA', 'SMR', 'MU', 'MRVL', 'MSFT', 'ASML', 'AEM', 'AMD', 'VERU', 'AI', 'GOOGL', 'INGM', 'PLUG', 'IONQ', 'CHYM', 'RGTI', 'ARBE']


Assets:   6%|██                                  | 1/18 [00:08<02:31,  8.89s/it]

⚠️ RDDT: query failed: Your default credentials were not found. To set up Application Default Credentials, see https://cloud.google.com/docs/authentication/external/set-up-adc for more information.


Assets:  11%|████                                | 2/18 [00:11<01:26,  5.43s/it]

⚠️ NVDA: query failed: Your default credentials were not found. To set up Application Default Credentials, see https://cloud.google.com/docs/authentication/external/set-up-adc for more information.


...

⚠️ MRVL: query failed: Your default credentials were not found. To set up Application Default Credentials, see https://cloud.google.com/docs/authentication/external/set-up-adc for more information.


Assets:  33%|████████████                        | 6/18 [00:24<00:40,  3.38s/it]

⚠️ MSFT: query failed: Your default credentials were not found. To set up Application Default Credentials, see https://cloud.google.com/docs/authentication/external/set-up-adc for more information.


Assets:  33%|████████████                        | 6/18 [00:28<00:57,  4.81s/it]


KeyboardInterrupt: 

In [15]:
from google.colab import auth
auth.authenticate_user()


ModuleNotFoundError: No module named 'google.colab'